# RV University - School of Computer Science and Engineering
## CS2235 - Data Science Capstone Project
### Title: Cryptocurrency Risk Analysis using K-Means Clustering

**Team Members:**
1. **Thanisha S** - USN: `1RUA25SCS0110`
2. **Srujana Shreya Raju** - USN: `1RUA25SCS0105`
3. **Rachana B** - USN: `1RUA25SCS0079`

**Program:** B.Sc. (Hons.) Data Science, Semester III  
**Course Code:** CS2235 - Data Science  

---

### Project Objective:
In this project, we analyze cryptocurrency market data to group digital currencies into distinct **risk categories** based on two main characteristics:
1. **Price Volatility (%)**: How much the price of the coin changes/fluctuates over time.
2. **Trading Volume (USD)**: How actively the coin is traded in the market.

We use **Unsupervised Machine Learning (K-Means Clustering)** following the exact laboratory methods taught in CS2235 (Feature Scaling with StandardScaler, K-Means Clustering, Silhouette Score evaluation, and Centroid plotting).


## Phase 1: Problem Identification (Clustering vs Classification vs Regression)

Before writing code, we identify the machine learning type:
- **Regression**: Used when predicting continuous numbers (e.g., predicting the exact price of Bitcoin tomorrow).
- **Classification**: Used when predicting predefined categories using labeled historical data (e.g., spam vs not spam).
- **Clustering (Unsupervised)**: Used when we want to discover natural groupings without having predefined labels.

Since cryptocurrencies do not have predefined official risk labels, **Clustering (K-Means)** is the correct and appropriate machine learning technique for our task.

In [ ]:
# Importing standard beginner data science libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

print("All libraries imported successfully!")

## Phase 2: Data Loading & Preprocessing

Here we load the dataset `crypto_data.csv`, inspect the first few rows, check for missing values, and display summary statistics.

In [ ]:
# Step 1: Load the dataset
try:
    df = pd.read_csv('crypto_data.csv')
    print("Dataset loaded successfully!")
    print("\n--- First 5 Rows ---")
    print(df.head())
except FileNotFoundError:
    print("File not found. Please make sure 'crypto_data.csv' is uploaded.")

In [ ]:
# Step 2: Check dataset information and missing values
print("--- Dataset Shape ---")
print(df.shape)

print("\n--- Missing Values Check ---")
print(df.isnull().sum())

print("\n--- Summary Statistics ---")
print(df.describe())

## Phase 3: Feature Selection & Feature Scaling

As done in **Lab Program 9 (K-Means Clustering on Mall Customers)**:
1. We select the numerical features for clustering: `Volatility_Percentage` and `Trading_Volume_USD`.
2. We use `StandardScaler()` so both features have mean = 0 and variance = 1. This prevents the large dollar volume from overshadowing the percentage volatility.

In [ ]:
# Step 3: Feature Selection
# Selecting the two key numerical features
X = df[['Volatility_Percentage', 'Trading_Volume_USD']]

# Handle missing values if any (using fillna with mean as in lab manual)
X = X.fillna(X.mean())

print("Selected Features Head:")
print(X.head())

In [ ]:
# Step 4: Feature Scaling using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled Features (First 5 Rows):")
print(X_scaled[:5])

## Phase 4: K-Means Clustering & Finding Optimal K (Elbow Method)

We determine the best number of clusters ($k$) by plotting the **Elbow Curve** (Inertia / Within-Cluster Sum of Squares) for values of $k$ from 2 to 7.

In [ ]:
# Step 5: Finding optimal K using the Elbow Method
wcss = []
k_range = range(2, 8)

for k in k_range:
    kmeans_test = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_test.fit(X_scaled)
    wcss.append(kmeans_test.inertia_)

# Plotting the Elbow Curve
plt.figure(figsize=(7, 4))
plt.plot(k_range, wcss, marker='o', color='b', linestyle='--')
plt.title('Elbow Method for Optimal K')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia (WCSS)')
plt.grid(True)
plt.show()

In [ ]:
# Step 6: Apply K-Means with chosen k=4
k = 4
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

# Add cluster results back to the original dataframe
df['Cluster'] = clusters

# Step 7: Evaluate the clustering using Silhouette Score
sil_score = silhouette_score(X_scaled, clusters)
print(f"Silhouette Score for k={k}: {sil_score:.4f}")

## Phase 5: Cluster Visualization & Risk Interpretation

We visualize the clusters and plot the **Centroids** (marked with a red 'X'), exactly as practiced in **Lab Program 9**.

In [ ]:
# Step 8: Visualize the Clusters with Centroids (Lab Manual Style)
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=clusters, cmap='viridis', s=60, alpha=0.8)

# Plot cluster centers (centroids)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], 
            c='red', marker='x', s=200, label='Centroids', linewidths=3)

plt.title('K-Means Clustering of Cryptocurrencies by Risk')
plt.xlabel('Volatility (Scaled)')
plt.ylabel('Trading Volume (Scaled)')
plt.legend()
plt.grid(True)
plt.colorbar(scatter, label='Cluster ID')
plt.show()

In [ ]:
# Step 9: Interpreting each Cluster (Summary Table)
cluster_summary = df.groupby('Cluster').agg({
    'Volatility_Percentage': 'mean',
    'Trading_Volume_USD': 'mean',
    'Symbol': 'count'
}).rename(columns={'Symbol': 'Number_of_Coins'})

print("--- Cluster Summary Statistics ---")
print(cluster_summary)

print("\n--- Coins in Each Cluster ---")
for c in sorted(df['Cluster'].unique()):
    coins = df[df['Cluster'] == c]['Symbol'].tolist()
    print(f"Cluster {c}: {', '.join(coins)}")

In [ ]:
# Step 10: Assigning Human-Readable Risk Tiers
# Based on the average volatility of each cluster:
# Cluster with lowest volatility -> Low Risk
# Cluster with next lowest volatility -> Moderate Risk
# Cluster with higher volatility -> High Risk
# Cluster with highest volatility -> Extreme Speculative Risk

vol_order = df.groupby('Cluster')['Volatility_Percentage'].mean().sort_values().index.tolist()
tier_names = ['Tier 1: Low Risk (Stable)', 'Tier 2: Moderate Risk (Large Cap)', 'Tier 3: High Risk (Mid Cap)', 'Tier 4: Extreme Risk (Meme/Speculative)']
cluster_to_risk = {cluster_id: tier_name for cluster_id, tier_name in zip(vol_order, tier_names)}

df['Risk_Category'] = df['Cluster'].map(cluster_to_risk)

print("--- Final Results with Risk Categories ---")
print(df[['Symbol', 'Name', 'Volatility_Percentage', 'Risk_Category']].head(15))

## Phase 6: Predicting Risk Category for New / Unseen Data

As in **Lab Program 9 (Step 8: Predict for new customer data)**, we can now take a new cryptocurrency with given volatility and volume, scale it, and predict its risk tier.

In [ ]:
# Step 11: Predict risk for a new cryptocurrency
# Example: A new meme coin with 130% volatility and $500M trading volume
new_crypto = pd.DataFrame({
    'Volatility_Percentage': [130.0],
    'Trading_Volume_USD': [500000000]
})

# Scale new data using the fitted scaler
new_crypto_scaled = scaler.transform(new_crypto)

# Predict cluster
predicted_cluster = kmeans.predict(new_crypto_scaled)[0]
predicted_tier = cluster_to_risk[predicted_cluster]

print(f"New Coin Volatility: 130%, Volume: $500,000,000")
print(f"Predicted Cluster: {predicted_cluster}")
print(f"Assigned Risk Tier: {predicted_tier}")

## Conclusion & Key Insights

1. **Successful Grouping**: K-Means clustering separated the 26 cryptocurrencies into 4 distinct, meaningful risk tiers.
2. **Stablecoins vs Meme Coins**: Stablecoins (USDT, USDC) clearly fell into the lowest risk cluster due to near-zero volatility, while meme coins (BONK, WIF, DOGE, SHIB) fell into the highest risk cluster.
3. **Practical Utility**: This simple, unsupervised model allows investors to understand the relative risk of cryptocurrencies without needing manual, subjective labels.